### Session 在哪里创建？使用结束后又在哪里关闭？
yield 依赖要解决的问题

### 普通 return 依赖的限制
普通依赖可以返回资源,但函数执行到 return 就结束了，因此无法在路由使用完成后继续执行清理代码。

所以资源型依赖通常不能只用 return。

### yield 把依赖分为进入和退出两部分
```python
async def get_session():
    session = create_session()

    try:
        yield session
    finally:
        await session.close()
```
进入时创建资源，退出时清理资源。

### 依赖的执行顺序
1. 创建 Session
2. yield Session
3. 路由使用 Session
4. 路由执行结束
5. 回到依赖的 yield 后面
6. 关闭 Session


> yield 不是返回后彻底结束，而是：
> 暂停依赖函数，把资源交给路由；路由结束后再恢复依赖函数。

### 同步资源与异步资源

In [ ]:
# 同步资源对应同步清理
def get_file():
    file = open("data.txt")

    try:
        yield file
    finally:
        file.close()

In [ ]:
# 异步资源对应异步清理
async def get_session():
    session = create_async_session()

    try:
        yield session
    finally:
        await session.close()

In [ ]:
# 如果资源提供异步上下文管理器可以更简单
# 离开 async with 时，会自动执行 Session 的退出逻辑。
async def get_session():
    async with session_factory() as session:
        yield session

In [ ]:
### 依赖中捕获异常要小心
async def get_session():
    session = create_session()

    try:
        yield session
    except Exception:
        await session.rollback()
        raise
    finally:
        await session.close()
# 注意raise，如果捕获异常后不再抛出，可能让异常传播变得不符合预期，使真正的业务错误被隐藏

## 面试
> FastAPI 的 yield 依赖适合管理数据库 Session 等请求级资源。依赖在 yield 前创建资源，将 yield 的值注入路由，路由结束后恢复执行并在 finally 中释放资源。即使路由抛出异常，清理逻辑仍会运行。多个 yield 子依赖会按创建顺序的反方向退出，类似嵌套上下文管理器。